# Project 1: Deep and Bayesian Reinforcement Learning for Variational Monte Carlo, deadline March 31st, 2027

## Introduction

The aim of this project is to extend a standard variational Monte Carlo study of a two-electron quantum dot by introducing reinforcement learning and Bayesian or uncertainty-aware reinforcement learning as tools for controlling the numerical optimization.

The underlying physics problem is the same as in the ordinary variational Monte Carlo formulation: we study two interacting electrons confined to a two-dimensional harmonic oscillator trap. The electrons interact through a repulsive Coulomb interaction, and the goal is to approximate the ground-state energy of the system.

In oscillator units, we set

$$
\hbar = m = e = 1.
$$

The Hamiltonian is

$$
\hat H
=
\sum_{i=1}^{2}
\left(
-\frac{1}{2}\nabla_i^2
+
\frac{1}{2}\omega^2 r_i^2
\right)
+
\frac{1}{r_{12}},
$$

where

$$
r_i = |\mathbf r_i|,
\qquad
r_{12}=|\mathbf r_1-\mathbf r_2|.
$$

Here $\omega$ is the oscillator frequency, $r_i$ is the distance of electron $i$ from the origin, and $r_{12}$ is the distance between the two electrons.

The variational principle states that for any trial wave function $\Psi_T$, the expectation value

$$
E[\Psi_T]
=
\frac{
\langle \Psi_T | \hat H | \Psi_T\rangle
}{
\langle \Psi_T | \Psi_T\rangle
}
$$

is an upper bound to the exact ground-state energy $E_0$,

$$
E_0 \leq E[\Psi_T].
$$

The task in variational Monte Carlo is therefore to choose a parametrized trial wave function and optimize its parameters so that the expectation value of the energy is minimized.

A simple trial wave function for the two-electron quantum dot is the Gaussian ansatz

$$
\Psi_T(\mathbf R;\alpha)
=
\exp
\left[
-\frac{\alpha\omega}{2}
\left(
r_1^2+r_2^2
\right)
\right],
$$

where

$$
\mathbf R = (\mathbf r_1,\mathbf r_2)
$$

is the full electronic configuration, and $\alpha$ is a variational parameter.

A more accurate trial wave function includes an explicit correlation factor between the two electrons. We use the Pade-Jastrow form

$$
\Psi_T(\mathbf R;\alpha,\beta)
=
\exp
\left[
-\frac{\alpha\omega}{2}
\left(
r_1^2+r_2^2
\right)
+
\frac{a r_{12}}{1+\beta r_{12}}
\right].
$$

The parameter $\alpha$ controls the spatial extent of the single-particle Gaussian part, while $\beta$ controls the range and strength of the two-body correlation factor. The constant $a$ is fixed by the cusp condition or by the convention used in the implementation. In this project, $a$ should be treated as fixed, while $\alpha$ and $\beta$ are variational parameters.

The local energy is defined as

$$
E_L(\mathbf R)
=
\frac{1}{\Psi_T(\mathbf R)}
\hat H
\Psi_T(\mathbf R).
$$

The variational energy can be written as an expectation value over the probability distribution

$$
p_\theta(\mathbf R)
=
\frac{
|\Psi_T(\mathbf R;\theta)|^2
}{
\int d\mathbf R |\Psi_T(\mathbf R;\theta)|^2
},
$$

where

$$
\theta = \alpha
$$

for the Gaussian ansatz, and

$$
\theta = (\alpha,\beta)
$$

for the Pade-Jastrow ansatz.

Thus,

$$
E(\theta)
=
\int d\mathbf R\,
p_\theta(\mathbf R)
E_L(\mathbf R).
$$

In Monte Carlo form this becomes

$$
E(\theta)
\approx
\frac{1}{M}
\sum_{k=1}^{M}
E_L(\mathbf R_k),
$$

where the configurations $\mathbf R_k$ are sampled from $|\Psi_T(\mathbf R;\theta)|^2$.

The variance of the local energy is

$$
\sigma_E^2
=
\langle E_L^2\rangle
-
\langle E_L\rangle^2.
$$

This quantity is important because an exact eigenstate has constant local energy and therefore zero variance. The energy and variance together give information about the quality of the trial wave function.

Sampling may first be done with the ordinary Metropolis algorithm. Starting from a configuration $\mathbf R$, propose

$$
\mathbf R'
=
\mathbf R+\delta\boldsymbol\xi,
$$

where $\boldsymbol\xi$ is a random displacement and $\delta$ is the proposal step length. The move is accepted with probability

$$
A(\mathbf R\rightarrow\mathbf R')
=
\min
\left[
1,
\frac{
|\Psi_T(\mathbf R')|^2
}{
|\Psi_T(\mathbf R)|^2
}
\right].
$$

An improved version uses importance sampling. In that case, proposed moves are generated using a drift-diffusion process,

$$
\mathbf R'
=
\mathbf R
+
D\mathbf F_Q(\mathbf R)\Delta t
+
\sqrt{\Delta t}\,\boldsymbol\eta,
$$

where

$$
D=\frac{1}{2},
$$

$\Delta t$ is a time step, $\boldsymbol\eta$ is a normally distributed random vector, and

$$
\mathbf F_Q(\mathbf R)
=
2\nabla_{\mathbf R}\ln|\Psi_T(\mathbf R)|
$$

is the quantum force.

The Metropolis-Hastings acceptance probability is then

$$
A(\mathbf R\rightarrow\mathbf R')
=
\min
\left[
1,
\frac{
G(\mathbf R'\rightarrow\mathbf R)
|\Psi_T(\mathbf R')|^2
}{
G(\mathbf R\rightarrow\mathbf R')
|\Psi_T(\mathbf R)|^2
}
\right],
$$

where $G$ is the Green's function associated with the drift-diffusion proposal.

For evaluating the kinetic energy, it is useful to write

$$
\frac{1}{\Psi_T}
\nabla_i^2\Psi_T
=
\nabla_i^2\ln\Psi_T
+
\left(
\nabla_i\ln\Psi_T
\right)^2.
$$

Then the local energy can be written as

$$
E_L(\mathbf R)
=
-\frac{1}{2}
\sum_{i=1}^{2}
\left[
\nabla_i^2\ln\Psi_T
+
\left(
\nabla_i\ln\Psi_T
\right)^2
\right]
+
V(\mathbf R),
$$

where

$$
V(\mathbf R)
=
\frac{1}{2}\omega^2(r_1^2+r_2^2)
+
\frac{1}{r_{12}}.
$$

For the Pade-Jastrow factor, define

$$
u(r)
=
\frac{a r}{1+\beta r}.
$$

Then

$$
\ln\Psi_T
=
-\frac{\alpha\omega}{2}
(r_1^2+r_2^2)
+
u(r_{12}),
$$

with

$$
u'(r)
=
\frac{a}{(1+\beta r)^2},
$$

and

$$
u''(r)
=
-\frac{2a\beta}{(1+\beta r)^3}.
$$

These expressions are useful for computing local energies, gradients, and quantum forces.

The new element in this project is that the optimization and numerical control of the VMC calculation will be formulated as a reinforcement-learning problem. The reinforcement-learning agent does not replace the VMC calculation. Instead, it controls selected numerical choices inside the calculation.

For example, the agent may control

$$
\alpha,
\qquad
\beta,
\qquad
\delta,
\qquad
\Delta t,
$$

or a subset of these. Here $\alpha$ and $\beta$ are variational parameters, while $\delta$ and $\Delta t$ control the Monte Carlo sampling.

A single reinforcement-learning step corresponds to one block of VMC sampling. At step $t$, the agent observes a state vector containing information about the current status of the calculation, for example

$$
S_t
=
\left(
\hat E_t,
\hat\sigma_{E,t}^2,
A_{\mathrm{acc},t},
\alpha_t,
\beta_t,
\delta_t,
\Delta t_t,
\hat E_t-\hat E_{t-1},
t/T
\right).
$$

Here $\hat E_t$ is the estimated energy in the current Monte Carlo block, $\hat\sigma_{E,t}^2$ is the estimated local-energy variance, $A_{\mathrm{acc},t}$ is the acceptance rate, and $T$ is the total number of reinforcement-learning steps in one episode.

The action space may be chosen as a discrete set of parameter updates, for example

$$
\mathcal A
=
\{
\alpha\uparrow,
\alpha\downarrow,
\beta\uparrow,
\beta\downarrow,
\delta\uparrow,
\delta\downarrow,
\Delta t\uparrow,
\Delta t\downarrow,
\text{do nothing}
\}.
$$

After an action has been selected, the corresponding parameter is changed, a new VMC block is performed, and the agent receives a reward. A simple reward is

$$
R_{t+1}
=
\hat E_t-\hat E_{t+1}.
$$

This rewards the agent when the energy decreases. A more stable reward may also include penalties for large variance and poor sampling,

$$
R_{t+1}
=
(\hat E_t-\hat E_{t+1})
-
\lambda_\sigma
\hat\sigma_{E,t+1}^2
-
\lambda_A
(A_{\mathrm{acc},t+1}-A^\star)^2.
$$

Here $A^\star$ is a target acceptance rate, for example

$$
A^\star = 0.5.
$$

The first reinforcement-learning approach in this project is classical deep reinforcement learning. A deep Q-network, for example, approximates the action-value function

$$
Q(s,a)
\approx
Q_\phi(s,a),
$$

where $\phi$ denotes the neural-network parameters. The agent uses this function to estimate which action is best in a given state. Alternatively, one may use a policy-gradient or actor-critic method, where the policy is parametrized directly as

$$
\pi_\theta(a\mid s).
$$

In both cases, the classical deep RL agent learns point estimates of values, policies, or both.

The second approach is Bayesian or uncertainty-aware reinforcement learning. The motivation is that VMC estimates are noisy and Monte Carlo samples are expensive. A standard deep RL agent may learn that an action appears good or bad, but it does not automatically know how uncertain that estimate is.

A Bayesian reinforcement-learning viewpoint introduces uncertainty over unknown quantities. Instead of only learning a single value estimate

$$
Q(s,a),
$$

one may think in terms of a belief distribution

$$
P(Q(s,a)\mid D_t),
$$

where $D_t$ denotes the data collected up to time $t$. Similarly, one may consider uncertainty over models,

$$
P(M\mid D_t),
$$

where $M$ represents a possible environment model.

Exact Bayesian deep reinforcement learning is usually computationally demanding. In this project, it is sufficient to use a practical uncertainty-aware approximation. A recommended choice is an ensemble of value networks,

$$
Q_1(s,a),
Q_2(s,a),
\ldots,
Q_K(s,a).
$$

The ensemble mean is

$$
\mu_Q(s,a)
=
\frac{1}{K}
\sum_{k=1}^{K}
Q_k(s,a),
$$

and the ensemble standard deviation is

$$
\sigma_Q(s,a)
=
\operatorname{Std}
\left(
Q_1(s,a),\ldots,Q_K(s,a)
\right).
$$

The mean gives a value estimate, while the spread gives an approximate uncertainty estimate. This uncertainty can be used for exploration. For example, an optimistic uncertainty-aware agent may choose

$$
A_t
=
\arg\max_a
\left[
\mu_Q(S_t,a)
+
\lambda_U\sigma_Q(S_t,a)
\right].
$$

Alternatively, a Thompson-style agent may sample one network from the ensemble at the beginning of an episode and act greedily with respect to that sampled network.

The project therefore compares three layers of methodology on the same quantum-mechanical problem:

$$
\text{ordinary VMC optimization},
$$

$$
\text{classical deep reinforcement learning},
$$

and

$$
\text{Bayesian or uncertainty-aware reinforcement learning}.
$$

The central question is not simply whether reinforcement learning can be made to run. The central question is whether reinforcement learning, and especially uncertainty-aware reinforcement learning, can make better decisions when controlling a noisy and expensive variational Monte Carlo calculation.

The project should investigate whether the learned agents can find good variational parameters, whether they can control the sampling efficiently, whether they outperform simple baselines, and whether uncertainty estimates lead to more robust exploration. A successful project should combine physical interpretation with algorithmic analysis. The final discussion should explain not only which method gives the lowest energy, but also why the method behaves as it does.

### a) Variational Monte Carlo for the two-electron quantum dot

Write a Variational Monte Carlo program for two electrons in a two-dimensional harmonic oscillator trap with Hamiltonian

$$
\hat H
=
\sum_{i=1}^{2}
\left(
-\frac{1}{2}\nabla_i^2
+
\frac{1}{2}\omega^2 r_i^2
\right)
+
\frac{1}{r_{12}}.
$$

Use oscillator units with

$$
\hbar=m=e=1,
$$

and set initially

$$
\omega=1.
$$

Use the trial wave function

$$
\Psi_T(\mathbf R;\alpha,\beta)
=
\exp
\left[
-\frac{\alpha\omega}{2}
(r_1^2+r_2^2)
+
\frac{a r_{12}}{1+\beta r_{12}}
\right],
$$

where

$$
\mathbf R=(\mathbf r_1,\mathbf r_2),
\qquad
r_{12}=|\mathbf r_1-\mathbf r_2|.
$$

Treat $a$ as fixed and use $\alpha$ and $\beta$ as variational parameters.

Implement the local energy

$$
E_L(\mathbf R)
=
\frac{1}{\Psi_T(\mathbf R)}
\hat H\Psi_T(\mathbf R),
$$

and estimate the expectation value

$$
E(\alpha,\beta)
=
\frac{1}{M}
\sum_{k=1}^{M}
E_L(\mathbf R_k),
$$

where the configurations $\mathbf R_k$ are sampled from

$$
|\Psi_T(\mathbf R;\alpha,\beta)|^2.
$$

Use the Metropolis algorithm with proposal

$$
\mathbf R'
=
\mathbf R+\delta\boldsymbol\xi,
$$

and acceptance probability

$$
A(\mathbf R\rightarrow\mathbf R')
=
\min
\left[
1,
\frac{
|\Psi_T(\mathbf R')|^2
}{
|\Psi_T(\mathbf R)|^2
}
\right].
$$

Compute, for each run, the estimated energy, the variance

$$
\sigma_E^2
=
\langle E_L^2\rangle
-
\langle E_L\rangle^2,
$$

and the acceptance rate.

Make a plot of the energy as a function of $\alpha$ for fixed $\beta$, and a contour plot of the energy in the $(\alpha,\beta)$ plane. Use these plots to identify a reasonable variational minimum.

Repeat the calculation for several values of the Metropolis step length $\delta$ and show how the acceptance rate depends on $\delta$.

Your answer to this part should contain:

- the expression used for the trial wave function;
- the expression or algorithm used for the local energy;
- plots of $E(\alpha)$ and $E(\alpha,\beta)$;
- a plot of acceptance rate versus $\delta$;
- the best values of $\alpha$ and $\beta$ found in your scan;
- the corresponding energy and variance.

The purpose of this part is to obtain a reliable VMC baseline before reinforcement learning is introduced.

### c) Formulate the VMC calculation as a reinforcement-learning problem

Use the VMC solver from part (a) as an environment in the reinforcement-learning sense. One RL step should correspond to one block of Monte Carlo sampling.

At step $t$, the agent observes a state $S_t$, chooses an action $A_t$, receives a reward $R_{t+1}$, and observes a new state $S_{t+1}$. This follows the agent-environment interaction

$$
S_t \rightarrow A_t \rightarrow (R_{t+1},S_{t+1}).
$$

Use a state vector containing the current VMC diagnostics, for example

$$
S_t =
\left(
\hat E_t,
\hat\sigma_{E,t}^2,
A_{\mathrm{acc},t},
\alpha_t,
\beta_t,
\delta_t,
\hat E_t-\hat E_{t-1},
t/T
\right).
$$

Here $\hat E_t$ is the estimated energy, $\hat\sigma_{E,t}^2$ is the variance of the local energy, $A_{\mathrm{acc},t}$ is the acceptance rate, and $T$ is the maximum number of RL steps in one episode.

Use a discrete action space. For example,

$$
\mathcal A =
\{
\alpha\uparrow,
\alpha\downarrow,
\beta\uparrow,
\beta\downarrow,
\delta\uparrow,
\delta\downarrow,
\text{do nothing}
\}.
$$

The actions should change the VMC parameters by fixed increments, for example

$$
\alpha \leftarrow \alpha \pm \Delta \alpha,
\qquad
\beta \leftarrow \beta \pm \Delta \beta,
\qquad
\delta \leftarrow \delta \pm \Delta \delta.
$$

Keep all parameters inside predefined bounds.

Define the reward so that the agent is encouraged to lower the energy while keeping the Monte Carlo sampling stable. Use

$$
R_{t+1}
=
(\hat E_t-\hat E_{t+1})
-
\lambda_\sigma \hat\sigma_{E,t+1}^2
-
\lambda_A
(A_{\mathrm{acc},t+1}-A^\star)^2.
$$

Use

$$
A^\star = 0.5
$$

unless you have a reason to choose another target acceptance rate.

Implement the environment with the standard structure

```python
state = env.reset()
next_state, reward, done, info = env.step(action)

### d) Deep Q-learning for VMC control

Train a Deep Q-learning agent for the environment defined in part (c).

The agent should approximate the action-value function

$$
q(s,a) \approx \hat q(s,a;w),
$$

where $w$ denotes the neural-network parameters. The input to the network is the state $S_t$, and the output is one action value for each allowed action.

Use the one-step Q-learning target

$$
Y_t
=
R_{t+1}
+
\gamma
\max_a
\hat q(S_{t+1},a;w^-),
$$

where $w^-$ denotes the target-network parameters. Train the network by minimizing

$$
L(w)
=
\left[
Y_t-\hat q(S_t,A_t;w)
\right]^2.
$$

Use experience replay and a target network. Choose actions during training with an $\epsilon$-greedy policy.

Train the agent for several episodes. Plot

$$
G_0
=
\sum_{t=0}^{T-1}
\gamma^t R_{t+1},
$$

as a function of episode number. Also plot the final energy obtained at the end of each episode.

After training, evaluate the learned policy without exploration. Compare the final energy, variance, and acceptance rate with the baselines from part (b).

Your answer should include:

- the neural-network architecture used for $\hat q(s,a;w)$;
- the values of $\gamma$, learning rate, replay-buffer size, and $\epsilon$ schedule;
- a plot of return versus episode;
- a plot of final energy versus episode;
- the final learned trajectory of $\alpha$, $\beta$, and $\delta$;
- a comparison with the best non-RL baseline.

The purpose of this part is to test whether a standard deep RL agent can learn useful control actions for the VMC calculation.

### e) Bayesian reinforcement learning through uncertainty-aware Q-learning

Extend the Deep Q-learning agent from part (d) to include uncertainty in the action-value estimates.

Use an ensemble of $K$ Q-networks,

$$
\hat q_1(s,a;w_1),
\hat q_2(s,a;w_2),
\ldots,
\hat q_K(s,a;w_K).
$$

For each state-action pair, define the ensemble mean

$$
\mu_q(s,a)
=
\frac{1}{K}
\sum_{k=1}^{K}
\hat q_k(s,a;w_k),
$$

and the ensemble standard deviation

$$
\sigma_q(s,a)
=
\operatorname{Std}
\left[
\hat q_1(s,a;w_1),
\ldots,
\hat q_K(s,a;w_K)
\right].
$$

Use the ensemble mean as the value estimate and the ensemble spread as an estimate of uncertainty.

Choose actions during training using either optimistic exploration,

$$
A_t
=
\arg\max_a
\left[
\mu_q(S_t,a)
+
\lambda_U\sigma_q(S_t,a)
\right],
$$

or Thompson-style exploration, where one network from the ensemble is chosen at the beginning of each episode and followed greedily during that episode.

Train the uncertainty-aware agent on the same environment, with the same state space, action space, reward function, and sampling budget as in part (d).

Compare the Bayesian or uncertainty-aware agent with the ordinary DQN agent.

Your answer should include:

- the ensemble size $K$;
- the exploration rule used;
- the value of $\lambda_U$, if using optimistic exploration;
- a plot of return versus episode;
- a plot of final energy versus episode;
- a comparison of final energy, variance, and acceptance rate with ordinary DQN;
- a plot or discussion of where the ensemble uncertainty is largest.

The purpose of this part is to test whether uncertainty-aware exploration improves learning when VMC estimates are noisy and expensive.

### f) Comparison of ordinary and uncertainty-aware reinforcement learning

Compare the methods from parts (b), (d), and (e) using the same total Monte Carlo sampling budget.

For each method, run several independent trials with different random seeds. Report the final energy, local-energy variance, and acceptance rate,

$$
\hat E,
\qquad
\hat\sigma_E^2,
\qquad
A_{\mathrm{acc}}.
$$

Include a table comparing:

- the best non-RL baseline from part (b);
- the ordinary Deep Q-learning agent from part (d);
- the uncertainty-aware Q-learning agent from part (e).

Use the same evaluation protocol for both RL agents. After training, freeze the learned policy and run evaluation episodes without additional random exploration.

Plot the final energy distribution over random seeds for all methods. Also plot one representative trajectory of

$$
\alpha_t,
\qquad
\beta_t,
\qquad
\delta_t,
\qquad
\hat E_t
$$

for the ordinary DQN agent and for the uncertainty-aware agent.

Discuss whether the uncertainty-aware agent improves over ordinary DQN. In particular, address whether it finds lower energies, uses fewer samples, gives more stable results across seeds, or explores the parameter space more efficiently.

The final discussion should answer the main scientific question:

**Does representing uncertainty help when reinforcement learning is used to control a noisy Variational Monte Carlo calculation?**

### g) Optional extension: RL for quantum state preparation

As an optional extension, apply the same ordinary-versus-uncertainty-aware RL comparison to a simple quantum-control problem.

Consider a two-level quantum system with Hamiltonian

$$
\hat H(t)
=
\frac{\Delta}{2}\sigma_z
+
\frac{\Omega(t)}{2}\sigma_x,
$$

where $\Omega(t)$ is a controllable field. The goal is to drive the system from an initial state $|\psi_0\rangle$ to a target state $|\psi_{\mathrm{target}}\rangle$.

Use the fidelity

$$
F_T
=
|\langle \psi_{\mathrm{target}}|\psi(T)\rangle|^2
$$

as the final performance measure.

Formulate this as an RL problem. A possible state is

$$
S_t =
\left(
\langle\sigma_x\rangle_t,
\langle\sigma_y\rangle_t,
\langle\sigma_z\rangle_t,
\Omega_t,
t/T
\right).
$$

Use a discrete action space,

$$
\mathcal A
=
\{
\Omega\uparrow,
\Omega\downarrow,
\text{do nothing}
\}.
$$

For the reward, use either a terminal reward

$$
R_T = F_T,
$$

or a shaped reward

$$
R_{t+1}
=
F_{t+1}-F_t
-
\lambda_\Omega \Omega_t^2.
$$

Train one ordinary DQN agent and one uncertainty-aware ensemble DQN agent. Compare their final fidelities over several random seeds.

Report:

- the target state used;
- the time horizon $T$;
- the final fidelity distribution;
- one learned control pulse $\Omega(t)$;
- whether uncertainty-aware exploration improves state preparation.

The purpose of this optional part is to show that the same RL ideas used for VMC optimization can also be applied to quantum engineering and control.

## Literature
Something here

## Introduction to numerical projects
Here follows a brief recipe and recommendation on how to write a report for each project.

- Give a short description of the nature of the problem and the eventual numerical methods you have used.

- Describe the algorithm you have used and/or developed. Here you may find it convenient to use pseudocoding. In many cases you can describe the algorithm in the program itself.

- Include the source code of your program. Comment your program properly.

- If possible, try to find analytic solutions, or known limits in order to test your program when developing the code.

- Include your results either in figure form or in a table. Remember to label your results. All tables and figures should have relevant captions and labels on the axes.

- Try to evaluate the reliabilty and numerical stability/precision of your results. If possible, include a qualitative and/or quantitative discussion of the numerical stability, eventual loss of precision etc.

- Try to give an interpretation of you results in your answers to the problems.

- - Critique: if possible include your comments and reflections about the exercise, whether you felt you learnt something, ideas for improvements and other thoughts you've made when solving the exercise. We wish to keep this course at the interactive level and your comments can help us improve it.

- Try to establish a practice where you log your work at the computerlab. You may find such a logbook very handy at later stages in your work, especially when you don't properly remember what a previous test version of your program did. Here you could also record the time spent on solving the exercise, various algorithms you may have tested or other topics which you feel worthy of mentioning.

## Format for electronic delivery of report and programs
The preferred format for the report is a PDF file. You can also use DOC or postscript formats or as an ipython notebook file. As programming language we prefer that you choose between C/C++, Fortran2008 or Python. The following prescription should be followed when preparing the report:

- Use canvas to hand in your projects, log in at http://canvas.uio.no with your normal UiO username and password.

- Upload only the report file! For the source code file(s) you have developed please provide us with your link to your github domain. The report file should include all of your discussions and a list of the codes you have developed. The full version of the codes should be in your github repository.

- In your github repository, please include a folder which contains selected results. These can be in the form of output from your code for a selected set of runs and input parameters.

- Still in your github make a folder where you place your codes.

- In this and all later projects, you should include tests (for example unit tests) of your code(s).

- Comments from us on your projects, approval or not, corrections to be made etc can be found under your Devilry domain and are only visible to you and the teachers of the course.

- Finally, we encourage you to work two and two together. Optimal working groups consist of 2-3 students. You can then hand in a common report.